# LeetCode 1921 — Eliminate Maximum Number of Monsters bằng Python

Link đề: https://leetcode.com/problems/eliminate-maximum-number-of-monsters/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def eliminateMaximum(self, dist: List[int], speed: List[int]) -> int:
        ...
```

Lộ trình: **thử mọi thứ tự bắn** (hoán vị, mô phỏng từng phút) → **mô phỏng tham lam: mỗi phút bắn con đến sớm nhất** `O(n²)` → **sắp xếp thời điểm đến, so với phút bắn** `O(n log n)` (lời giải nộp) → **đếm theo phút (counting sort)** `O(n)`.

## 1. Mục tiêu học

1. Chuyển bài toán vật lý (khoảng cách, vận tốc) thành **mốc thời gian đến** `tᵢ = distᵢ / speedᵢ` — chỉ đại lượng này quyết định.
2. Hiểu lập luận tham lam **"bắn con đến sớm nhất trước"** (earliest deadline first) và chứng minh bằng **trao đổi**.
3. Xử lý chính xác **biên bằng nhau** (quái đến đúng lúc súng nạp xong là **thua**) và **làm tròn lên** bằng số nguyên: `⌈d/s⌉ = (d + s − 1) // s`.
4. Tối ưu từ sắp xếp `O(n log n)` xuống **đếm theo phút** `O(n)` nhờ chỉ quan tâm phút `< n`.
5. Kiểm chứng bằng vét cạn mọi thứ tự bắn trên input ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Bạn bảo vệ thành phố khỏi `n` quái vật. `dist[i]` là khoảng cách ban đầu (km) của quái `i` tới thành phố, `speed[i]` là vận tốc (km/phút) — quái đi thẳng tới với vận tốc không đổi.

Vũ khí khi **nạp đầy** tiêu diệt được **một** quái; nạp mất **1 phút**. Vũ khí đầy ngay lúc bắt đầu.

Bạn **thua** khi có quái tới thành phố. Nếu quái tới **đúng lúc** vũ khí vừa nạp xong thì vẫn tính là **thua** (trò chơi kết thúc trước khi kịp bắn).

Trả về số quái **nhiều nhất** bạn tiêu diệt được trước khi thua, hoặc `n` nếu tiêu diệt hết.

**Ví dụ 1:** `dist = [1,3,4], speed = [1,1,1]` → `3`.

**Ví dụ 2:** `dist = [1,1,2,3], speed = [1,1,1,1]` → `1` (sau phút đầu, quái thứ hai đã tới: khoảng cách `[X,0,1,2]`).

**Ví dụ 3:** `dist = [3,2,4], speed = [5,3,2]` → `1`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ dist[i], speed[i] ≤ 10⁵`.

In [1]:
import random
import time
from fractions import Fraction
from itertools import permutations
from typing import List

examples = [([1, 3, 4], [1, 1, 1], 3), ([1, 1, 2, 3], [1, 1, 1, 1], 1), ([3, 2, 4], [5, 3, 2], 1)]
for d, s, want in examples:
    print("dist =", d, "speed =", s, "-> kỳ vọng", want)

dist = [1, 3, 4] speed = [1, 1, 1] -> kỳ vọng 3
dist = [1, 1, 2, 3] speed = [1, 1, 1, 1] -> kỳ vọng 1
dist = [3, 2, 4] speed = [5, 3, 2] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1 Thời điểm đến và phân số chính xác

Quái `i` tới thành phố tại `tᵢ = distᵢ / speedᵢ` phút (có thể không nguyên). `fractions.Fraction` biểu diễn phân số **chính xác**, tiện để kiểm chứng.

In [2]:
for d, s in [(3, 5), (2, 3), (4, 2)]:
    print(f"d={d}, s={s}: t = {Fraction(d, s)} ≈ {d / s:.3f} phút")

d=3, s=5: t = 3/5 ≈ 0.600 phút
d=2, s=3: t = 2/3 ≈ 0.667 phút
d=4, s=2: t = 2 ≈ 2.000 phút


### 3.2 Làm tròn lên bằng số nguyên

`⌈d / s⌉ = (d + s − 1) // s` với `d, s > 0` — không cần số thực. Tương đương `-(-d // s)`.

In [3]:
for d, s in [(3, 5), (4, 2), (5, 2), (10, 3)]:
    print(f"ceil({d}/{s}) = {(d + s - 1) // s} = {-(-d // s)}")

ceil(3/5) = 1 = 1
ceil(4/2) = 2 = 2
ceil(5/2) = 3 = 3
ceil(10/3) = 4 = 4


### 3.3 Phút bắn

Lần bắn thứ `j` (đếm từ 0) diễn ra tại phút `j`: phút 0 (vũ khí đầy sẵn), phút 1, phút 2, … Bắn được quái `i` tại phút `j` ⇔ quái chưa tới: `tᵢ > j` (nghiêm ngặt, vì tới đúng phút `j` là thua).

In [4]:
t = Fraction(2, 1)
for j in range(4):
    print(f"phút {j}: quái đến lúc t={t} còn bắn được? {t > j}")

phút 0: quái đến lúc t=2 còn bắn được? True
phút 1: quái đến lúc t=2 còn bắn được? True
phút 2: quái đến lúc t=2 còn bắn được? False
phút 3: quái đến lúc t=2 còn bắn được? False


### 3.4 `sorted` và `enumerate`

Sắp xếp các thời điểm đến tăng dần, rồi so phần tử thứ `j` với phút `j`.

In [5]:
times = sorted([Fraction(3, 5), Fraction(2, 3), Fraction(2, 1)])
for j, tt in enumerate(times):
    print(j, tt, "ok" if tt > j else "THUA")

0 3/5 ok
1 2/3 THUA
2 2 THUA


### 3.5 Đếm theo "xô" (counting sort)

Nếu chỉ quan tâm giá trị nhỏ (ở đây thời điểm làm tròn lên `< n`), có thể đếm số phần tử mỗi giá trị bằng mảng `cnt` rồi duyệt tăng dần — `O(n)`.

In [6]:
vals, n = [3, 1, 1, 7, 2], 5
cnt = [0] * (n + 1)
for v in vals:
    cnt[min(v, n)] += 1                 # mọi giá trị >= n gộp vào xô n
print(cnt)

[0, 2, 1, 1, 0, 1]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 3: `dist = [3,2,4]`, `speed = [5,3,2]` ⇒ thời điểm đến `t = [0.6, 0.667, 2]`.

- Phút 0: bắn được 1 con. Hai con còn lại đến lúc 0.6 và 0.667 — **trước** phút 1.
- Phút 1: vũ khí nạp xong nhưng đã có quái tới (lúc 0.6) ⇒ thua. Kết quả **1**.

Ví dụ 2: `t = [1, 1, 2, 3]`. Phút 0 bắn một con `t = 1`. Phút 1: con còn lại có `t = 1` tới **đúng** lúc nạp xong ⇒ thua. Kết quả **1**.

Nhận xét: khoảng cách lớn chưa chắc an toàn (con `dist = 3` đến trước con `dist = 2` vì nhanh hơn). Thứ quan trọng là **thời điểm đến**; và nên bắn con **đến sớm nhất** trước.

In [7]:
for d, s, want in examples:
    print([str(Fraction(a, b)) for a, b in zip(d, s)], "-> đáp án", want)

['1', '3', '4'] -> đáp án 3
['1', '1', '2', '3'] -> đáp án 1
['3/5', '2/3', '2'] -> đáp án 1


## 5. Cách 1 — Brute Force / baseline: thử mọi thứ tự bắn

**Ý tưởng:** thử mọi hoán vị thứ tự bắn; với mỗi thứ tự, lần bắn thứ `j` (phút `j`) hợp lệ nếu **mọi** quái chưa bị bắn đều chưa tới (`t > j`). Đếm số lần bắn tới khi thua, lấy max.

In [8]:
def eliminate_brute(dist, speed):
    n = len(dist)
    t = [Fraction(d, s) for d, s in zip(dist, speed)]
    best = 0
    for order in permutations(range(n)):
        alive = set(range(n))
        killed = 0
        for j, idx in enumerate(order):
            if any(t[a] <= j for a in alive):      # có quái đã tới (hoặc tới đúng phút j)
                break
            alive.remove(idx); killed += 1
        best = max(best, killed)
    return best

for d, s, want in examples:
    print(eliminate_brute(d, s), "| kỳ vọng", want)

3 | kỳ vọng 3
1 | kỳ vọng 1
1 | kỳ vọng 1


**Độ phức tạp:** `n!` thứ tự × `O(n²)` mô phỏng ⇒ chỉ dùng `n ≤ 8`. **Nút thắt:** hầu hết thứ tự hiển nhiên tệ (bắn con ở xa trong khi con gần sắp tới) ⇒ cần lập luận chọn **một** thứ tự tốt nhất.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — chỉ thời điểm đến quan trọng:** quái `i` "hết hạn" lúc `tᵢ`; bắn nó tại phút `j` hợp lệ khi `j < tᵢ`.

**Insight 2 — tham lam theo hạn chót:** sắp xếp `t₍₀₎ ≤ t₍₁₎ ≤ …`. Bắn theo thứ tự này; tại phút `j` ta bắn con có hạn thứ `j`. Thua ngay tại phút `j` đầu tiên mà `t₍ⱼ₎ ≤ j` (con đó — và có thể các con khác — đã tới).

**Quy tắc:** đáp án = chỉ số `j` nhỏ nhất với `t₍ⱼ₎ ≤ j`; nếu không có thì `n`.

**Điều kiện bằng số nguyên:** `tᵢ ≤ j ⇔ distᵢ ≤ j · speedᵢ ⇔ ⌈distᵢ / speedᵢ⌉ ≤ j`. Đặt `aᵢ = ⌈distᵢ / speedᵢ⌉` — "phút đầu tiên mà quái đã ở thành phố" — thì có thể làm việc hoàn toàn bằng số nguyên.

**Invariant:** nếu đã bắn `j` con theo thứ tự hạn tăng mà chưa thua, thì `j` con đó là `j` con có hạn sớm nhất, và mọi con còn lại có hạn `≥ t₍ⱼ₎`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng tham lam từng phút (`O(n²)`)

In [9]:
def eliminate_sim(dist, speed):
    alive = [(Fraction(d, s), i) for i, (d, s) in enumerate(zip(dist, speed))]
    minute = 0
    while alive:
        if min(alive)[0] <= minute:               # có quái đã tới
            return minute
        alive.remove(min(alive))                  # bắn con đến sớm nhất
        minute += 1
    return len(dist)

for d, s, want in examples:
    print(eliminate_sim(d, s), want)

3 3
1 1
1 1


**Độ phức tạp:** mỗi phút tìm `min` `O(n)` ⇒ `O(n²)`. Nút thắt: tìm lại min mỗi phút — sắp xếp **một lần** là đủ (thứ tự hạn không đổi theo thời gian).

### 7.2 Cách 3 — Sắp xếp thời điểm đến (làm tròn lên) (`O(n log n)`) — lời giải chuẩn

In [10]:
def eliminate_sort_demo(dist, speed):
    arrive = sorted((d + s - 1) // s for d, s in zip(dist, speed))
    print("phút đầu tiên quái đã ở thành phố (đã sắp xếp):", arrive)
    for j, a in enumerate(arrive):
        if a <= j:
            print(f"phút {j}: quái có a={a} <= {j} -> thua, đã bắn {j}")
            return j
        print(f"phút {j}: bắn quái a={a} (> {j})")
    return len(arrive)

print("=>", eliminate_sort_demo([3, 2, 4], [5, 3, 2]))

phút đầu tiên quái đã ở thành phố (đã sắp xếp): [1, 1, 2]
phút 0: bắn quái a=1 (> 0)
phút 1: quái có a=1 <= 1 -> thua, đã bắn 1
=> 1


In [11]:
class SolutionSort:
    def eliminateMaximum(self, dist: List[int], speed: List[int]) -> int:
        arrive = sorted((d + s - 1) // s for d, s in zip(dist, speed))
        for j, a in enumerate(arrive):
            if a <= j:
                return j
        return len(arrive)

for d, s, want in examples:
    print(SolutionSort().eliminateMaximum(d, s), want)

3 3
1 1
1 1


**Độ phức tạp:** `O(n log n)` sắp xếp + `O(n)` duyệt; bộ nhớ `O(n)`.

### 7.3 Cách 4 — Đếm theo phút (`O(n)`)

Chỉ cần biết, với mỗi phút `j < n`, có bao nhiêu quái có `aᵢ = j`. Quái có `aᵢ ≥ n` không bao giờ gây thua trước khi ta bắn hết. Duyệt `j = 0..n−1`, cộng dồn số quái có `a ≤ j`; thua tại `j` đầu tiên mà số đó **vượt** `j` (tức có hơn `j` con phải bắn trước phút `j`).

In [12]:
class SolutionCount:
    def eliminateMaximum(self, dist: List[int], speed: List[int]) -> int:
        n = len(dist)
        cnt = [0] * (n + 1)
        for d, s in zip(dist, speed):
            a = (d + s - 1) // s
            if a < n:
                cnt[a] += 1
        arrived = 0                       # số quái có a <= j
        for j in range(n):
            arrived += cnt[j]
            if arrived > j:               # cần đã bắn > j con trước phút j, nhưng chỉ bắn được j
                return j
        return n

for d, s, want in examples:
    print(SolutionCount().eliminateMaximum(d, s), want)

3 3
1 1
1 1


**Vì sao `arrived > j` ⇔ `a₍ⱼ₎ ≤ j`?** Trong dãy đã sắp xếp, `a₍ⱼ₎ ≤ j` ⇔ có ít nhất `j + 1` phần tử `≤ j` ⇔ `arrived(j) ≥ j + 1`. Và phút thua đầu tiên giống nhau. **Độ phức tạp:** `O(n)` thời gian và bộ nhớ.

### 7.4 Cách 5 — Heap theo thời điểm đến (mô phỏng "online")

Nếu quái **xuất hiện dần** (dữ liệu đến theo thời gian) thì không sắp xếp trước được; khi đó dùng min-heap các thời điểm đến và mỗi phút `heappop` con sớm nhất. Với bài này kết quả y hệt Cách 3, nhưng khung code này dùng lại được cho biến thể "quái xuất hiện ở các phút khác nhau".

In [13]:
import heapq

class SolutionHeap:
    def eliminateMaximum(self, dist: List[int], speed: List[int]) -> int:
        h = [(d + s - 1) // s for d, s in zip(dist, speed)]
        heapq.heapify(h)                      # O(n)
        minute = 0
        while h:
            if h[0] <= minute:                # con sớm nhất đã tới
                return minute
            heapq.heappop(h)                  # bắn nó
            minute += 1
        return minute

for d, s, want in examples:
    print(SolutionHeap().eliminateMaximum(d, s), want)

3 3
1 1
1 1


**Độ phức tạp:** `heapify` `O(n)` + tối đa `n` lần `heappop` `O(log n)` ⇒ `O(n log n)` worst, nhưng nếu thua sớm ở phút `J` thì chỉ `O(n + J log n)` — nhanh hơn sắp xếp toàn bộ khi `J` nhỏ.

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 1 và Ví dụ 2:

| ví dụ | `aᵢ = ⌈d/s⌉` sắp xếp | phút 0 | phút 1 | phút 2 | phút 3 | đáp án |
|---|---|---|---|---|---|---|
| 1 | `[1, 3, 4]` | 1 > 0 ✔ | 3 > 1 ✔ | 4 > 2 ✔ | — | **3** |
| 2 | `[1, 1, 2, 3]` | 1 > 0 ✔ | 1 ≤ 1 ✘ | | | **1** |

Trace Cách 4 trên Ví dụ 2: `cnt = [0, 2, 1, 1, 0]`; `j = 0`: `arrived = 0` ≤ 0 ✔; `j = 1`: `arrived = 2 > 1` ⇒ thua, trả 1.

In [14]:
def trace_count(dist, speed):
    n = len(dist); cnt = [0] * (n + 1)
    for d, s in zip(dist, speed):
        a = (d + s - 1) // s
        cnt[min(a, n)] += 1
    print("cnt (xô n gộp mọi a >= n):", cnt)
    arrived = 0
    for j in range(n):
        arrived += cnt[j]
        print(f"phút {j}: số quái đã/đang tới (a <= {j}) = {arrived} {'> ' if arrived > j else '<='} {j}")
        if arrived > j:
            return j
    return n

for d, s, want in examples:
    print("=>", trace_count(d, s), "| kỳ vọng", want); print()

cnt (xô n gộp mọi a >= n): [0, 1, 0, 2]
phút 0: số quái đã/đang tới (a <= 0) = 0 <= 0
phút 1: số quái đã/đang tới (a <= 1) = 1 <= 1
phút 2: số quái đã/đang tới (a <= 2) = 1 <= 2
=> 3 | kỳ vọng 3

cnt (xô n gộp mọi a >= n): [0, 2, 1, 1, 0]
phút 0: số quái đã/đang tới (a <= 0) = 0 <= 0
phút 1: số quái đã/đang tới (a <= 1) = 2 >  1
=> 1 | kỳ vọng 1

cnt (xô n gộp mọi a >= n): [0, 2, 1, 0]
phút 0: số quái đã/đang tới (a <= 0) = 0 <= 0
phút 1: số quái đã/đang tới (a <= 1) = 2 >  1
=> 1 | kỳ vọng 1



## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def eliminateMaximum(self, dist: List[int], speed: List[int]) -> int:
        # a = phút đầu tiên (số nguyên) mà quái đã ở thành phố = ceil(dist / speed)
        arrive = sorted((d + s - 1) // s for d, s in zip(dist, speed))
        for minute, a in enumerate(arrive):
            if a <= minute:          # quái tới trước hoặc đúng lúc bắn lần thứ `minute`
                return minute
        return len(arrive)

for d, s, want in examples:
    assert Solution().eliminateMaximum(d, s) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Chuyển sang số nguyên.** Bắn quái `i` ở phút `j` hợp lệ ⇔ `tᵢ > j` ⇔ `distᵢ > j·speedᵢ` ⇔ `distᵢ / speedᵢ > j` ⇔ `⌈distᵢ / speedᵢ⌉ > j` (vì `j` nguyên). Nên làm việc với `aᵢ = ⌈distᵢ/speedᵢ⌉`.

**Cận trên (không ai làm tốt hơn).** Gọi `a₍₀₎ ≤ a₍₁₎ ≤ …` đã sắp xếp và `J` là chỉ số nhỏ nhất với `a₍J₎ ≤ J`. Có `J + 1` quái với `a ≤ a₍J₎ ≤ J` — tức mỗi con phải bị bắn ở một phút `< a ≤ J`, nghĩa là trong các phút `0..J−1` — chỉ có `J` phút. Theo nguyên lý chuồng bồ câu, ít nhất một con không bị bắn kịp và tới thành phố ở thời điểm `≤ J` ⇒ không thể bắn quá `J` con. (Nếu không có `J`, cận trên hiển nhiên là `n`.)

**Đạt được.** Bắn theo thứ tự `a` tăng: tại phút `j < J`, con bị bắn có `a₍ⱼ₎ > j` ✔, và mọi con còn lại có `a ≥ a₍ⱼ₎ > j` nên chưa tới ⇒ không thua trước phút `J`. Vậy đạt đúng `J` (hoặc `n`).

Đây chính là lập luận **trao đổi** của tham lam "hạn chót sớm nhất trước": mọi lịch tối ưu có thể hoán đổi để bắn con hạn sớm hơn trước mà không xấu đi.

Minh họa lập luận chuồng bồ câu bằng máy: tại phút thua `J`, đếm số quái có `a ≤ J` — luôn **lớn hơn** `J` (số phút bắn được trước đó), nên mọi lịch bắn đều để lọt ít nhất một con.

In [16]:
rng_p = random.Random(55)
for _ in range(500):
    n = rng_p.randint(1, 30)
    d = [rng_p.randint(1, 40) for _ in range(n)]; s = [rng_p.randint(1, 10) for _ in range(n)]
    J = Solution().eliminateMaximum(d, s)
    if J < n:
        must = sum(1 for a, b in zip(d, s) if (a + b - 1) // b <= J)
        assert must > J
print("tại phút thua J: số quái phải bắn trước phút J luôn > J")

tại phút thua J: số quái phải bắn trước phút J luôn > J


## 11. Phân tích Time / Space

| Cách | Time | Space |
|---|---|---|
| 1. Mọi thứ tự | `O(n! · n²)` | `O(n)` |
| 2. Mô phỏng tham lam | `O(n²)` | `O(n)` |
| 3. Sắp xếp | `O(n log n)` | `O(n)` |
| 4. Đếm theo phút | `O(n)` | `O(n)` |
| 5. Heap | `O(n + J log n)` (`J` = đáp án) | `O(n)` |

In [17]:
rng = random.Random(1921)
n = 10**5
d1 = [rng.randint(1, 10**5) for _ in range(n)]; s1 = [rng.randint(1, 10**5) for _ in range(n)]
d2 = [10**5 - i for i in range(n)]; s2 = [1] * n                    # thời điểm đến đều > chỉ số: bắn hết
for label, (d, s) in [("ngẫu nhiên", (d1, s1)), ("bắn hết", (d2, s2))]:
    res = []
    for name, sol in [("Sắp xếp/Nộp", Solution()), ("Đếm", SolutionCount())]:
        t = time.perf_counter(); r = sol.eliminateMaximum(d, s); res.append(r)
        print(f"[{label}] {name}: {r} trong {1000*(time.perf_counter()-t):.0f} ms")
    assert len(set(res)) == 1
t = time.perf_counter(); eliminate_sim(d2[:1500], s2[:1500]); dt = time.perf_counter() - t
print(f"Cách 2 với n=1500: {dt:.2f}s -> n=10^5 ước tính ≈ {dt * (n / 1500) ** 2:.0f}s")

[ngẫu nhiên] Sắp xếp/Nộp: 1 trong 10 ms
[ngẫu nhiên] Đếm: 1 trong 7 ms
[bắn hết] Sắp xếp/Nộp: 100000 trong 9 ms
[bắn hết] Đếm: 100000 trong 10 ms


Cách 2 với n=1500: 0.97s -> n=10^5 ước tính ≈ 4330s


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu vét cạn)

In [18]:
sols = [Solution(), SolutionSort(), SolutionCount(), SolutionHeap()]
edge = examples + [
    ([1], [1], 1),                       # bắn ngay phút 0
    ([1, 1], [1, 1], 1),                 # con thứ hai tới đúng phút 1
    ([2, 2], [1, 1], 2),
    ([3, 3], [2, 2], 2),                 # t = 1.5 > 1: kịp
    ([5, 5, 5], [5, 5, 5], 1),           # t = 1 cho cả ba
    ([10, 1, 100], [1, 1, 50], 3),       # t = 10, 1, 2 -> bắn theo thứ tự 1, 2, 10
    ([4, 2, 3], [2, 1, 1], 3),
]
for d, s, want in edge:
    assert eliminate_brute(d, s) == want, (d, s, eliminate_brute(d, s))
    for sol in sols:
        assert sol.eliminateMaximum(d, s) == want, (d, s, sol)
    assert eliminate_sim(d, s) == want
    print("OK", d, s, "->", want)

OK [1, 3, 4] [1, 1, 1] -> 3
OK [1, 1, 2, 3] [1, 1, 1, 1] -> 1
OK [3, 2, 4] [5, 3, 2] -> 1
OK [1] [1] -> 1
OK [1, 1] [1, 1] -> 1
OK [2, 2] [1, 1] -> 2
OK [3, 3] [2, 2] -> 2
OK [5, 5, 5] [5, 5, 5] -> 1
OK [10, 1, 100] [1, 1, 50] -> 3
OK [4, 2, 3] [2, 1, 1] -> 3


### 12.2 Ngẫu nhiên có seed: so với vét cạn mọi thứ tự

In [19]:
rng = random.Random(42)
for _ in range(800):
    n = rng.randint(1, 6)
    hi = rng.choice([3, 6, 10])
    d = [rng.randint(1, hi) for _ in range(n)]
    s = [rng.randint(1, hi) for _ in range(n)]
    want = eliminate_brute(d, s)
    for sol in sols:
        assert sol.eliminateMaximum(d, s) == want, (d, s)
    assert eliminate_sim(d, s) == want
for _ in range(50):
    n = rng.randint(1, 2000)
    d = [rng.randint(1, 10**5) for _ in range(n)]; s = [rng.randint(1, 10**5) for _ in range(n)]
    want = eliminate_sim(d, s)
    assert all(sol.eliminateMaximum(d, s) == want for sol in sols)
print("800 case nhỏ (vét cạn) + 50 case vừa (mô phỏng tham lam): khớp")

800 case nhỏ (vét cạn) + 50 case vừa (mô phỏng tham lam): khớp


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Làm tròn **xuống** (`d // s`) thay vì lên

In [20]:
def wrong_floor(dist, speed):
    arrive = sorted(d // s for d, s in zip(dist, speed))      # SAI
    for j, a in enumerate(arrive):
        if a <= j:
            return j
    return len(arrive)

for d, s in [([3, 3], [2, 2]), ([1, 3, 4], [1, 1, 1])]:
    print(d, s, "sai:", wrong_floor(d, s), "| đúng:", Solution().eliminateMaximum(d, s))

[3, 3] [2, 2] sai: 1 | đúng: 2
[1, 3, 4] [1, 1, 1] sai: 3 | đúng: 3


**Vì sao sai:** `t = 3/2 = 1.5` — quái tới **sau** phút 1, nên phút 1 vẫn bắn được. Làm tròn xuống thành 1 khiến ta tưởng nó tới đúng phút 1 ⇒ thua sớm. **Sửa:** `(d + s − 1) // s`.

### Lỗi 2 — Coi "tới đúng lúc nạp xong" là vẫn bắn được (`<` thay vì `≤`)

In [21]:
def wrong_strict(dist, speed):
    arrive = sorted((d + s - 1) // s for d, s in zip(dist, speed))
    for j, a in enumerate(arrive):
        if a < j:                                             # SAI: bằng cũng là thua
            return j
    return len(arrive)

for d, s, want in examples:
    print(d, s, "sai:", wrong_strict(d, s), "| đúng:", want)

[1, 3, 4] [1, 1, 1] sai: 3 | đúng: 3
[1, 1, 2, 3] [1, 1, 1, 1] sai: 4 | đúng: 1
[3, 2, 4] [5, 3, 2] sai: 3 | đúng: 1


**Vì sao sai:** đề nói rõ quái tới **đúng lúc** vũ khí nạp xong thì thua (Ví dụ 2). **Sửa:** `a <= j`.

### Lỗi 3 — Sắp xếp theo khoảng cách (bỏ qua vận tốc)

In [22]:
def wrong_by_dist(dist, speed):
    order = sorted(range(len(dist)), key=lambda i: dist[i])  # SAI: con xa nhưng nhanh vẫn tới sớm
    t = [Fraction(d, s) for d, s in zip(dist, speed)]
    alive = set(range(len(dist)))
    for j, i in enumerate(order):
        if any(t[a] <= j for a in alive):
            return j
        alive.remove(i)
    return len(dist)

for d, s in [([3, 2, 4], [5, 3, 2]), ([5, 1], [5, 1]), ([10, 2, 3], [10, 1, 1])]:
    print(d, s, "sai:", wrong_by_dist(d, s), "| đúng:", Solution().eliminateMaximum(d, s))

[3, 2, 4] [5, 3, 2] sai: 1 | đúng: 1
[5, 1] [5, 1] sai: 1 | đúng: 1
[10, 2, 3] [10, 1, 1] sai: 1 | đúng: 3


**Vì sao sai:** thứ tự bắn phải theo **thời điểm đến**. Ví dụ `dist = [10,2,3]`, `speed = [10,1,1]`: con xa nhất (10 km, 10 km/phút) tới sau 1 phút — phải bắn trước. (Phản ví dụ tự dựng, đã đối chiếu vét cạn qua mục 12.)

### Lỗi 4 — So sánh bằng số thực

In [23]:
d, s = 3, 3
print("3/3 =", d / s, "| phép so 1.0 <= 1:", d / s <= 1, "| ổn ở đây")
big_d, big_s = 10**17 + 1, 10**17
print("với số rất lớn: float:", big_d / big_s <= 1, "| chính xác:", Fraction(big_d, big_s) <= 1)

3/3 = 1.0 | phép so 1.0 <= 1: True | ổn ở đây
với số rất lớn: float: True | chính xác: False


**Vì sao nguy hiểm:** với ràng buộc `≤ 10⁵`, số thực double thường vẫn đúng, nhưng phép chia số thực có thể làm `1 + ε` thành `1.0` với số lớn (ngoài ràng buộc, minh họa ở trên) — sai đúng ở biên "bằng nhau" mà đề cố tình nhấn mạnh. **Sửa:** dùng số nguyên `⌈d/s⌉` hoặc so `d <= j * s`.

### Lỗi 5 — Mô phỏng từng phút nhưng bắn con **gần nhất** (theo khoảng cách hiện tại)

In [24]:
def wrong_nearest(dist, speed):
    pos = [Fraction(d) for d in dist]; alive = set(range(len(dist))); minute = 0
    while alive:
        if any(pos[i] <= 0 for i in alive):
            return minute
        i = min(alive, key=lambda a: pos[a])                 # SAI: gần nhất chưa chắc tới sớm nhất
        alive.remove(i); minute += 1
        for a in alive:
            pos[a] -= speed[a]
    return len(dist)

for d, s in [([4, 5], [1, 5]), ([10, 2, 3], [10, 1, 1])]:
    print(d, s, "sai:", wrong_nearest(d, s), "| đúng:", Solution().eliminateMaximum(d, s))

[4, 5] [1, 5] sai: 1 | đúng: 2
[10, 2, 3] [10, 1, 1] sai: 1 | đúng: 3


**Vì sao sai:** "gần nhất lúc này" khác "tới sớm nhất". `[4, 5]` với vận tốc `[1, 5]`: con 4 km đi chậm (tới sau 4 phút), con 5 km đi nhanh (tới sau 1 phút). Bắn con gần trước thì con nhanh tới đúng phút 1 ⇒ thua.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Mỗi đơn vị thời gian xử lý một việc, mỗi việc có hạn chót"** ⇒ sắp xếp theo hạn chót (EDF), kiểm `deadline₍ⱼ₎ > j`.
- Thời điểm là phân số ⇒ chuyển về **số nguyên**: `⌈d/s⌉` hoặc nhân chéo `d ≤ j·s`.
- Đọc kỹ biên "bằng nhau" (thua hay thắng?) — thường là bẫy chính.
- Giá trị chỉ quan trọng khi `< n` ⇒ counting sort cho `O(n)`.
- Bài tương tự: LeetCode 1353 (*Maximum Number of Events*), 630 (*Course Schedule III*), 2410.

## 15. Biến thể của bài

1. **Vũ khí nạp mất `c` phút** ⇒ lần bắn thứ `j` ở phút `j·c`; điều kiện `aᵢ` so với `j·c` (Bài tập 5).
2. **Mỗi lần bắn diệt `b` quái** ⇒ phút `j` diệt các con thứ `j·b … j·b + b − 1` trong thứ tự hạn.
3. **Quái có máu** (cần nhiều phát) ⇒ bài lập lịch có độ dài việc khác nhau — LeetCode 630 (heap).
4. **Bắt đầu với vũ khí chưa nạp** ⇒ lần bắn thứ `j` ở phút `j + 1`.
5. **Hỏi thời điểm thua** thay vì số con ⇒ chính là phút `J` (hoặc "không thua").

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi thứ tự | vét cạn | `O(n!·n²)` | `O(n)` | kiểm chứng |
| 2. Mô phỏng tham lam | min mỗi phút | `O(n²)` | `O(n)` | `n` nhỏ, trực quan |
| 3. Sắp xếp | `a₍ⱼ₎ ≤ j` đầu tiên | `O(n log n)` | `O(n)` | **nộp bài** |
| 4. Đếm theo phút | `arrived(j) > j` | `O(n)` | `O(n)` | tối ưu tuyến tính |
| 5. Heap | pop con sớm nhất mỗi phút | `O(n + J log n)` | `O(n)` | dữ liệu đến dần / thua sớm |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `([2,2],[1,1])`, `([1,1],[1,1])`, `([5],[10])`?

**Bài 2 (tính tay):** Tính `aᵢ = ⌈dᵢ/sᵢ⌉` cho `dist = [10,2,3]`, `speed = [10,1,1]`, sắp xếp và tìm đáp án.

**Bài 3 (điền TODO):** Viết `kill_order(dist, speed)` trả về danh sách **chỉ số** quái theo thứ tự bắn tối ưu, cắt tại lúc thua.

**Bài 4 (tự viết lại):** Viết lại lời giải **không** dùng phép chia: sắp xếp chỉ số theo `dist/speed` bằng so sánh nhân chéo (`functools.cmp_to_key`), rồi kiểm `dist ≤ j·speed`.

**Bài 5 (mini challenge):** Vũ khí nạp mất `c` phút (`c ≥ 1`, đầy sẵn lúc đầu). Viết `eliminate_with_charge(dist, speed, c)` và đối chiếu vét cạn.

In [25]:
def kill_order(dist, speed):
    # TODO
    pass

def eliminate_no_division(dist, speed):
    # TODO
    pass

def eliminate_with_charge(dist, speed, c):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Tính `a` | `[2,2]`: bắn hết | `[1,1]`: 1; `[5],[10]`: `a = 1 > 0` ⇒ 1 |
| 2 | `a = [1, 2, 3]` | phút 0: 1 > 0, phút 1: 2 > 1, phút 2: 3 > 2 | Đáp án 3 |
| 3 | Sắp xếp chỉ số theo `a` | Duyệt, dừng khi `a ≤ j` | Trả danh sách đã bắn |
| 4 | `d₁/s₁ < d₂/s₂ ⇔ d₁·s₂ < d₂·s₁` | `cmp_to_key` | Thua khi `d ≤ j·s` |
| 5 | Lần bắn `j` ở phút `j·c` | Thua khi `a₍ⱼ₎ ≤ j·c` | Vét cạn: mô phỏng với `j·c` |

## 19. Đáp án tham khảo

In [26]:
r1 = [Solution().eliminateMaximum(*a) for a in [([2, 2], [1, 1]), ([1, 1], [1, 1]), ([5], [10])]]
print("Bài 1:", r1); assert r1 == [2, 1, 1]
print("Bài 2:"); assert eliminate_sort_demo([10, 2, 3], [10, 1, 1]) == 3

Bài 1: [2, 1, 1]
Bài 2:
phút đầu tiên quái đã ở thành phố (đã sắp xếp): [1, 2, 3]
phút 0: bắn quái a=1 (> 0)
phút 1: bắn quái a=2 (> 1)
phút 2: bắn quái a=3 (> 2)


In [27]:
from functools import cmp_to_key

def kill_order(dist, speed):
    idx = sorted(range(len(dist)), key=lambda i: (dist[i] + speed[i] - 1) // speed[i])
    out = []
    for j, i in enumerate(idx):
        if (dist[i] + speed[i] - 1) // speed[i] <= j:
            break
        out.append(i)
    return out

def eliminate_no_division(dist, speed):
    def cmp(i, j):
        lhs, rhs = dist[i] * speed[j], dist[j] * speed[i]
        return -1 if lhs < rhs else (1 if lhs > rhs else 0)
    order = sorted(range(len(dist)), key=cmp_to_key(cmp))
    for j, i in enumerate(order):
        if dist[i] <= j * speed[i]:
            return j
    return len(dist)

rng = random.Random(3)
for _ in range(500):
    n = rng.randint(1, 30)
    d = [rng.randint(1, 50) for _ in range(n)]; s = [rng.randint(1, 50) for _ in range(n)]
    ko = kill_order(d, s)
    assert len(ko) == Solution().eliminateMaximum(d, s) == eliminate_no_division(d, s)
    t = [Fraction(d[i], s[i]) for i in range(n)]
    assert all(t[i] > j for j, i in enumerate(ko))           # mỗi lần bắn hợp lệ
print("Bài 3, 4: OK", kill_order([3, 2, 4], [5, 3, 2]), kill_order([1, 3, 4], [1, 1, 1]))

Bài 3, 4: OK [0] [0, 1, 2]


In [28]:
def eliminate_with_charge(dist, speed, c):
    arrive = sorted((d + s - 1) // s for d, s in zip(dist, speed))
    for j, a in enumerate(arrive):
        if a <= j * c:
            return j
    return len(arrive)

def charge_brute(dist, speed, c):
    n = len(dist); t = [Fraction(d, s) for d, s in zip(dist, speed)]; best = 0
    for order in permutations(range(n)):
        alive = set(range(n)); k = 0
        for j, idx in enumerate(order):
            if any(t[a] <= j * c for a in alive):
                break
            alive.remove(idx); k += 1
        best = max(best, k)
    return best

rng = random.Random(9)
for _ in range(400):
    n = rng.randint(1, 6); c = rng.randint(1, 3)
    d = [rng.randint(1, 12) for _ in range(n)]; s = [rng.randint(1, 4) for _ in range(n)]
    assert eliminate_with_charge(d, s, c) == charge_brute(d, s, c), (d, s, c)
    if c == 1:
        assert eliminate_with_charge(d, s, 1) == Solution().eliminateMaximum(d, s)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", eliminate_with_charge([1, 3, 4], [1, 1, 1], 2))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK 2


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
arrive = sorted((d + s - 1) // s for d, s in zip(dist, speed))   # ceil, số nguyên
for minute, a in enumerate(arrive):
    if a <= minute:            # tới trước hoặc ĐÚNG lúc bắn -> thua
        return minute
return len(arrive)
```

**Checklist:**
- [ ] Sắp xếp theo **thời điểm đến** `d/s`, không theo khoảng cách?
- [ ] Làm tròn **lên** `(d + s − 1) // s` (hoặc so `d <= j * s`)?
- [ ] Điều kiện thua `a <= minute` (bằng nhau là thua)?
- [ ] Bắn hết thì trả `n`?
- [ ] Không dùng số thực ở biên.